<a href="https://colab.research.google.com/github/rohandcs24-lgtm/AI-LAB/blob/main/1BM24CS240_Week6_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from collections import deque


def L(sign, predicate, *args):
    return (sign, predicate, args)


KB = [
    frozenset([
        L(False, "food", "?x"),
        L(True, "likes", "John", "?x")
    ]),

    frozenset([L(True, "food", "Apple")]),

    frozenset([L(True, "food", "Vegetables")]),

    frozenset([
        L(False, "eats", "?x", "?y"),
        L(True, "killed", "?x"),
        L(True, "food", "?y")
    ]),

    frozenset([L(True, "eats", "Anil", "Peanuts")]),

    frozenset([L(True, "alive", "Anil")]),

    frozenset([
        L(False, "eats", "Anil", "?x"),
        L(True, "eats", "Harry", "?x")
    ]),

    frozenset([
        L(False, "alive", "?x"),
        L(False, "killed", "?x")
    ]),

    frozenset([
        L(True, "killed", "?x"),
        L(True, "alive", "?x")
    ])
]


def dereference(term, subst):
    while term.startswith("?") and term in subst:
        term = subst[term]
    return term


def unify(t1, t2, subst):
    t1 = dereference(t1, subst)
    t2 = dereference(t2, subst)

    if t1 == t2:
        return subst

    if t1.startswith("?"):
        subst[t1] = t2
        return subst

    if t2.startswith("?"):
        subst[t2] = t1
        return subst

    return None


def apply_substitution(literal, subst):
    sign, predicate, args = literal

    new_args = tuple(
        dereference(arg, subst) for arg in args
    )

    return (sign, predicate, new_args)



counter = 0

def standardize(clause):
    global counter
    counter += 1

    mapping = {}

    for sign, pred, args in clause:
        for arg in args:
            if arg.startswith("?"):
                mapping[arg] = f"{arg}_{counter}"

    return frozenset(
        (
            sign,
            pred,
            tuple(mapping.get(arg, arg) for arg in args)
        )
        for sign, pred, args in clause
    )


def resolve(c1, c2):
    c1 = standardize(c1)
    c2 = standardize(c2)

    results = []

    for l1 in c1:
        for l2 in c2:

            sign1, pred1, args1 = l1
            sign2, pred2, args2 = l2

            if pred1 != pred2 or sign1 == sign2:
                continue

            if len(args1) != len(args2):
                continue

            subst = {}

            for a, b in zip(args1, args2):
                subst = unify(a, b, subst)

                if subst is None:
                    break

            if subst is None:
                continue

            remaining = (c1 - {l1}) | (c2 - {l2})

            new_clause = frozenset(
                apply_substitution(lit, subst)
                for lit in remaining
            )

            results.append(new_clause)

    return results



def show_clause(clause):
    if not clause:
        return "{}"

    parts = []

    for sign, pred, args in sorted(clause, key=str):
        text = pred + "(" + ", ".join(args) + ")"

        if not sign:
            text = "~" + text

        parts.append(text)

    return " OR ".join(parts)



def resolution(KB, query):

    negated_query = frozenset([
        L(False, "likes", "John", "Peanuts")
    ])

    clauses = list(KB) + [negated_query]

    support = deque([negated_query])

    visited = set(clauses)

    print("\nInitial Knowledge Base:")
    for i, clause in enumerate(clauses, 1):
        print(f"C{i}: {show_clause(clause)}")

    print("\nStarting Resolution...\n")

    steps = 0
    max_steps = 5000

    while support and steps < max_steps:

        current = support.popleft()
        for other in list(clauses):

            if current == other:
                continue

            resolvents = resolve(current, other)

            for new_clause in resolvents:

                if new_clause in visited:
                    continue

                visited.add(new_clause)
                clauses.append(new_clause)
                support.append(new_clause)

                steps += 1

                print(
                    f"R{steps}: {show_clause(new_clause)}"
                )

                if not new_clause:
                    print("\nEmpty clause {} obtained!")
                    print("Contradiction found.")
                    print("Query is PROVED.")
                    return True

                if steps >= max_steps:
                    break

            if steps >= max_steps:
                break

    print("\nNo contradiction found.")
    print("Query could not be proved.")
    return False




query = "likes(John, Peanuts)"

print("Query:", query)

result = resolution(KB, query)

if result:
    print("\nFINAL RESULT: John likes peanuts.")


Query: likes(John, Peanuts)

Initial Knowledge Base:
C1: ~food(?x) OR likes(John, ?x)
C2: food(Apple)
C3: food(Vegetables)
C4: ~eats(?x, ?y) OR food(?y) OR killed(?x)
C5: eats(Anil, Peanuts)
C6: alive(Anil)
C7: ~eats(Anil, ?x) OR eats(Harry, ?x)
C8: ~alive(?x) OR ~killed(?x)
C9: alive(?x) OR killed(?x)
C10: ~likes(John, Peanuts)

Starting Resolution...

R1: ~food(Peanuts)
R2: ~eats(?x_26, Peanuts) OR killed(?x_26)
R3: killed(Anil)
R4: ~eats(Anil, Peanuts) OR killed(Harry)
R5: ~alive(?x_54) OR ~eats(?x_54, Peanuts)
R6: ~alive(Anil)
R7: killed(Harry)
R8: ~alive(Harry) OR ~eats(Anil, Peanuts)
R9: ~eats(Anil, Peanuts)
R10: ~eats(?x_136, Peanuts) OR killed(?x_136)
R11: {}

Empty clause {} obtained!
Contradiction found.
Query is PROVED.

FINAL RESULT: John likes peanuts.
